# 11 综合盈利因子：研报原始权重版本复现

本 Notebook 是一个**权重口径扩展复现**。它复用既有四个底层子因子、数据时点、股票池、样本划分、收益标签和 RQAlpha 执行规则；唯一实质变化是综合盈利因子的合成权重。

研报第 7.2 节的两层规则为：

$$\text{利润表因子汇总}=(A+B+C)/3$$

$$\text{综合盈利因子}=0.5\times\text{利润表因子汇总}+0.5\times FCFFIC$$

因此四个底层子因子的展开权重是 $1/6,1/6,1/6,1/2$。当前项目仍使用既有重构/代理口径，不声称完成 Wind 原始字段的全量严格复现。

**内核顺序不可颠倒：** 11.1 在 Python 3.9 运行；11.2 切换到 Python 3.6 RQAlpha 运行；11.3 切回 Python 3.9。所有个股权重和 pkl 只保存在 `data/` 或 `cache/`，不可上传 GitHub。

In [ ]:
# 11.1 — 请确认当前 Kernel 是 Python 3.9，再运行本单元。
from pathlib import Path
import runpy

PROJECT_ROOT = Path('/home/rice/notebook/a-share-profitability-factor-replication')
assert PROJECT_ROOT.exists(), f'项目根目录不存在：{PROJECT_ROOT}'
runpy.run_path(str(PROJECT_ROOT / 'scripts' / '11_source_weight_prepare.py'))

## 11.2 RQAlpha 单组断点回测与期末清仓

切换到 Python 3.6 / RQAlpha Kernel 后，**每次只运行一个下方的 11.2 单元**。每组完成后会立刻保存私有可移植结果；如中断，已完成组会在后续运行时自动复用。不要点击“运行全部”，也不要在 3.6 中运行 11.1。

运行顺序：`base_q5`、`base_q1`、`base_equal`、`forced_q5`、`forced_q1`、`forced_equal`。看到“11.2 单组完成”后再运行下一格。

In [ ]:
# 11.2.0 — 请确认当前 Kernel 是 Python 3.6 / RQAlpha；先运行一次本单元。
from pathlib import Path
import os
import runpy

PROJECT_ROOT = Path('/home/rice/notebook/a-share-profitability-factor-replication')
assert PROJECT_ROOT.exists(), '项目根目录不存在：{}'.format(PROJECT_ROOT)

def run_rqalpha_task(task_name):
    os.environ['SOURCE_WEIGHT_RQALPHA_TASK'] = task_name
    runpy.run_path(str(PROJECT_ROOT / 'scripts' / '11_source_weight_rqalpha.py'))

print('已进入单组断点模式。请从 11.2.1 开始，每次只运行一格。')

In [ ]:
# 11.2.1 — 基础回测：Q5 高分组
run_rqalpha_task('base_q5')

In [ ]:
# 11.2.2 — 基础回测：Q1 低分组
run_rqalpha_task('base_q1')

In [ ]:
# 11.2.3 — 基础回测：可投资股票池等权基准组合
run_rqalpha_task('base_equal')

In [ ]:
# 11.2.4 — 强制清仓：Q5 高分组
run_rqalpha_task('forced_q5')

In [ ]:
# 11.2.5 — 强制清仓：Q1 低分组
run_rqalpha_task('forced_q1')

In [ ]:
# 11.2.6 — 强制清仓：可投资股票池等权基准组合
run_rqalpha_task('forced_equal')

## 11.3 聚合结果、强制清仓敏感性与结论规则

切回 Python 3.9。此步从可移植的私有结果读取逐日净值，重新计算 Q5 相对 Q1、相对可投资股票池等权基准组合的收益，并断言三个组合在清仓窗口后市值为零。

In [ ]:
# 11.3 — 请确认当前 Kernel 是 Python 3.9，再运行本单元。
from pathlib import Path
import runpy

PROJECT_ROOT = Path('/home/rice/notebook/a-share-profitability-factor-replication')
assert PROJECT_ROOT.exists(), f'项目根目录不存在：{PROJECT_ROOT}'
runpy.run_path(str(PROJECT_ROOT / 'scripts' / '11_source_weight_report.py'))

## 完成标准

- 11.1 输出的规则文件须写明两层公式和展开的 $1/6,1/6,1/6,1/2$ 权重，且程序会逐元素核对两种写法相等。
- 11.2 使用与旧版本相同的初始资金、CSI 800 基准、佣金、印花税、涨跌停、停牌、整手和 5% 成交量约束。
- 11.3 只在样本外 RankIC、Q5-Q1 的 HAC 证据和相对可投资股票池等权基准组合的表现共同支持时，才输出 `independent_alpha_supported`；否则为 `candidate_not_confirmed`。
- 本次是既有样本外结果被观察后开展的权重版本扩展复现，不能表述为首次确认性样本外检验。